# SQD Demo — N₂ / STO-3G

Demonstrates the full SQD pipeline on N₂ in the minimal STO-3G basis:
an inexpensive run that verifies sampling from the selected QPU and checks
agreement with the exact FCI result.
For a more demanding test of sampling quality, switch to the 6-31G notebook.

## Overview

The SQD algorithm combines:
1. **Quantum sampling** — LUCJ ansatz built by `ffsim.qiskit.lucj_pass_manager`,
   submitted to IBM Quantum via QRMI
2. **Classical post-processing** — configuration recovery + SBD diagonalization
   via `qiskit-addon-sqd`

## Credentials

Credentials are read from **environment variables**, never from config files.

1. Copy the example env file once:
   ```zsh
   cp examples/notebook_demos/sqd_demos/.env.example \
      examples/notebook_demos/sqd_demos/.env
   ```
2. Fill in `.env` with your IBM Quantum IAM API key and CRN:
   ```
   ibm_kingston_QRMI_IBM_QCS_IAM_APIKEY=<your-iam-api-key>
   ibm_kingston_QRMI_IBM_QCS_SERVICE_CRN=<your-crn>
   ```
   The prefix must be **lowercase** and match `QRMI_JOB_QPU_RESOURCES` exactly.

The `.env` file is gitignored — it will never be committed.

> **HPC / Slurm:** Skip the `.env` file. The Slurm SPANK plugin injects all
> `QRMI_*` variables automatically when the job requests a QPU resource.

## Setup

In [1]:
import os
import yaml
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from pyscf import gto, scf, fci, ao2mo

from quantum_fragment_methods.qpu import QRMIBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver

DEMO_DIR = Path('__file__').resolve().parent

## 1. Load Configuration

In [2]:
config_path = DEMO_DIR / 'config_N2_sto-3g_demo.yaml'

with open(config_path) as f:
    config = yaml.safe_load(f)

qpu_config = config['qpu']
sqd_config = config['sqd']

print(f'Config  : {config_path}')
print(f'Backend : {qpu_config["backend_name"]}')
print(f'Shots   : {qpu_config["sampler_options"]["default_shots"]}')
print(f'SQD iter: {sqd_config["iterations"]}  batches: {sqd_config["n_batches"]}  samples: {sqd_config["samples_per_batch"]}')

Config  : /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/config_N2_sto-3g_demo.yaml
Backend : ibm_kingston
Shots   : 100000
SQD iter: 3  batches: 1  samples: 200


## 2. Define Molecular System — N₂ / STO-3G

In [3]:
mol = gto.Mole()
mol.atom = '''
N 0.0 0.0 0.0
N 1.0 0.0 0.0
'''
mol.basis = 'sto-3g'
mol.build()

print(f'Orbitals : {mol.nao}')
print(f'Electrons: {mol.nelectron}')

Orbitals : 10
Electrons: 14


## 3. Hartree-Fock + MO-basis Hamiltonian

In [4]:
mf = scf.RHF(mol)
mf.kernel()
print(f'HF energy: {mf.e_tot:.8f}')

norb       = mol.nao
nelec      = (mol.nelec[0], mol.nelec[1])
mo_coeff   = mf.mo_coeff
h1e        = mo_coeff.T @ mf.get_hcore() @ mo_coeff
h2e        = ao2mo.restore(1, ao2mo.kernel(mol, mo_coeff), norb)
nuc_energy = mf.energy_nuc()

print(f'norb={norb}  nelec={nelec}  nuc={nuc_energy:.8f}')

converged SCF energy = -107.419532451682
HF energy: -107.41953245
norb=10  nelec=(7, 7)  nuc=25.92968334


## 4. Initialize QRMI Backend

Credentials are loaded from `.env` (or already set by Slurm). The backend
name is taken from `QRMI_JOB_QPU_RESOURCES` if set, otherwise falls back
to the value in the config file.

In [5]:
# Load .env without overriding variables already injected by Slurm SPANK plugin
load_dotenv(DEMO_DIR / '.env', override=False)

BACKEND_NAME = os.environ.get('QRMI_JOB_QPU_RESOURCES', qpu_config['backend_name'])
os.environ.setdefault('QRMI_JOB_QPU_RESOURCES', BACKEND_NAME)
os.environ.setdefault('QRMI_JOB_QPU_TYPES', 'ibm-quantum-compute-service')

backend = QRMIBackend({'backend_name': BACKEND_NAME})
backend.initialize()
backend.get_backend()

props = backend.get_backend_properties()
print(f'Backend: {props["backend_name"]}  ({props["resource_type"]})')

Backend: ibm_kingston  (ResourceType.IBMQuantumComputeService)


## 5. Run SQD

- Builds a LUCJ circuit via `ffsim.qiskit.lucj_pass_manager`
- Submits to IBM Quantum via QRMI
- Checkpoints to `sqd_results/` (re-running resumes from `counts.npy` if present)
- Runs SBD post-processing

In [6]:
solver       = SQDSolver(backend, config=sqd_config)
workflow_dir = DEMO_DIR / 'sqd_results' / 'N2_sto-3g'

result = solver.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=nelec,
    mf=mf,                        # CCSD amplitudes computed automatically
    workflow_path=str(workflow_dir),
    wait_for_completion=True,
    force_resubmit=False,          # set True to clear checkpoint and resubmit
)

print(f'Electronic energy: {result.energy:.8f} Ha')
print(f'Total energy     : {result.energy + nuc_energy:.8f} Ha')

E(CCSD) = -107.5467229235981  E_corr = -0.1271904719160964
  QPU job submitted: dai5ilfi3e6s738ol9p0
  Backend: ibm_kingston  checkpoint: /Users/thaddeuspellegrini/Code/QuantumAlgorithmEngineering/QDC_2026/quantum-fragment-methods/examples/notebook_demos/sqd_demos/sqd_results/N2_sto-3g/job_id.txt
  Polling job dai5ilfi3e6s738ol9p0 every 30s (max 300s) ...
  00s] dai5ilfi3e6s738ol9p0 — RUNNING
  30s] dai5ilfi3e6s738ol9p0 — RUNNING
  00s] dai5ilfi3e6s738ol9p0 — RUNNING
  30s] dai5ilfi3e6s738ol9p0 — DONE
  ✓ Job completed (1m30s)
  SQD: 3 iter × 1 batches × 200 samples  backend=python
  SBD complete — E = -133.47861455 Ha
Electronic energy: -133.47861455 Ha
Total energy     : -107.54893121 Ha


## 6. FCI Comparison

In [7]:
fci_solver = fci.FCI(mf)
fci_solver.verbose = 0
fci_solver.kernel()

sqd_total = result.energy + nuc_energy
sqd_error = abs(sqd_total - fci_solver.e_tot)

print(f'HF  energy : {mf.e_tot:.8f} Ha')
print(f'FCI energy : {fci_solver.e_tot:.8f} Ha')
print(f'SQD energy : {sqd_total:.8f} Ha')
print(f'SQD vs FCI : {sqd_error:.8f} Ha  ({sqd_error * 627.5:.4f} kcal/mol)')

HF  energy : -107.41953245 Ha
FCI energy : -107.54930096 Ha
SQD energy : -107.54893121 Ha
SQD vs FCI : 0.00036974 Ha  (0.2320 kcal/mol)
